# ChEMBL journal/DOI fingerprint degeneracy analysis

This notebook implements the following pipeline:

- Keep only **ChEMBL document/deposition records with a non-empty `journal` and a non-empty `doi`**. The DOI is not validated or resolved.
- Treat each ChEMBL document record as a deposition and use the assays linked to that deposition.
- Compare structures **only within the same deposition and the same assay**.
- Use ChEMBL's `compound_structures.canonical_smiles` **as-is**. No parent-fragment stripping or structure standardization is applied.
- Within each deposition/assay, collapse duplicate occurrences of the same SMILES. A fingerprint-degenerate pair therefore always contains **two different canonical SMILES strings**.
- Compute three exact bit fingerprints: Morgan radius 2, AtomPair, and TopologicalTorsion; all are 1024-bit and chirality-aware.
- A pair is fingerprint-degenerate when the two distinct SMILES have **bit-for-bit identical fingerprints** for a fingerprint type.
- To decide whether a degenerate pair also has an activity difference, only exact (`standard_relation = '='`) measurements with the same `standard_type` and `standard_units` are compared. Replicates for the same SMILES/assay/endpoint are collapsed to their median `standard_value` first.
- The final Plotly bar chart counts **unique DOI strings (papers)** that contain at least one degenerate pair with a non-zero comparable activity difference. The `Total / any FP` bar is the union across the three fingerprint types, not the sum of the other bars.

The expensive intermediate results are stored in a sidecar SQLite database so the workflow can be resumed without redoing completed steps.


## 1. Imports


In [1]:
from pathlib import Path
import sqlite3
import time
import matplotlib.pyplot as plt

import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display

import rdkit
from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator

pd.set_option("display.max_columns", 100)
print("RDKit:", rdkit.__version__)
print("pandas:", pd.__version__)


C:\Users\acebl\AppData\Local\Temp\ipykernel_27688\3047222012.py:7: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


RDKit: 2025.09.4
pandas: 2.2.0


## 2. Configuration
Edit the two paths below. `REBUILD=True` discards the cached sidecar analysis and starts again.


In [2]:
CHEMBL_DB = Path(r"C:\Users\acebl\Downloads\chembl_37_sqlite\chembl_37\chembl_37_sqlite\chembl_37.db")
OUTPUT_DIR = Path(r"C:\Users\acebl\Downloads\chembl_degeneracy_results_grouped")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

WORK_DB = OUTPUT_DIR / "journal_doi_fingerprint_degeneracy.sqlite"
REBUILD = False
FINGERPRINT_BATCH_SIZE = 5000

FP_NAMES = ["Morgan", "AtomPair", "TopologicalTorsion"]


## 3. Helpers and fingerprint generators


In [ ]:
def table_exists(conn, table_name):
    ''' Check whether a given table exists'''
    return conn.execute(
        "SELECT 1 FROM sqlite_master WHERE type='table' AND name=?",
        (table_name,),
    ).fetchone() is not None


def count_rows(conn, table_name):
    ''' Count the number of rows in a given table'''
    return conn.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]


def timed_sql(conn, sql, label):
    '''
        Track how long it takes for a query to execute 
        for logging purposes.
    '''
    t0 = time.time()
    conn.executescript(sql)
    conn.commit()
    print(f"{label}: {time.time() - t0:,.1f} s")


def attach_chembl_read_only(conn, db_path):
    if not db_path.exists():
        raise FileNotFoundError(f"ChEMBL database not found: {db_path}")
    # SQLite URI keeps the attached ChEMBL database read-only.
    uri = db_path.resolve().as_uri() + "?mode=ro"
    conn.execute("ATTACH DATABASE ? AS chembl", (uri,))

def fp_bytes(generator, mol):
    '''Binary representation of the 1024-bit fingerprint.'''
    return DataStructs.BitVectToBinaryText(generator.GetFingerprint(mol))

morgan_gen = rdFingerprintGenerator.GetMorganGenerator(
    radius=2, fpSize=1024, includeChirality=True
)
atompair_gen = rdFingerprintGenerator.GetAtomPairGenerator(
    fpSize=1024, includeChirality=True
)
torsion_gen = rdFingerprintGenerator.GetTopologicalTorsionGenerator(
    fpSize=1024, includeChirality=True
)

## 4. Open the sidecar database and check the ChEMBL schema


In [4]:
if REBUILD and WORK_DB.exists():
    WORK_DB.unlink()

work_uri = WORK_DB.expanduser().resolve().as_uri()
conn = sqlite3.connect(work_uri, uri=True)
conn.execute("PRAGMA journal_mode=WAL")
conn.execute("PRAGMA synchronous=NORMAL")
conn.execute("PRAGMA temp_store=FILE")
attach_chembl_read_only(conn, CHEMBL_DB)

required = {
    "docs": {"doc_id", "chembl_id", "doi", "journal"},
    "assays": {"assay_id", "doc_id", "chembl_id"},
    "activities": {
        "activity_id", "assay_id", "molregno", "standard_type",
        "standard_relation", "standard_value", "standard_units"
    },
    "molecule_dictionary": {"molregno", "chembl_id", "molecule_type"},
    "compound_structures": {"molregno", "canonical_smiles"},
}

for table, needed in required.items():
    have = {r[1] for r in conn.execute(f"PRAGMA chembl.table_info({table})")}
    missing = needed - have
    if missing:
        raise RuntimeError(f"chembl.{table} is missing columns: {sorted(missing)}")

print("Schema check passed.")


Schema check passed.


## 5. Extract journal + DOI depositions and their assay activities

The join runs through `docs → assays → activities`, which guarantees that comparisons remain tied to the assay associated with the deposition. Only small molecules with a non-empty ChEMBL canonical SMILES and a non-null standardized activity value are retained.


In [5]:
if not table_exists(conn, "relevant_activity"):
    timed_sql(conn, """
    CREATE TABLE relevant_activity AS
    SELECT
        d.doc_id,
        d.chembl_id AS paper_chembl_id,
        TRIM(d.doi) AS doi,
        d.journal,
        a.assay_id,
        a.chembl_id AS assay_chembl_id,
        act.activity_id,
        act.molregno,
        md.chembl_id AS molecule_chembl_id,
        cs.canonical_smiles,
        act.standard_type,
        act.standard_relation,
        act.standard_value,
        act.standard_units
    FROM chembl.docs AS d
    JOIN chembl.assays AS a
      ON a.doc_id = d.doc_id
    JOIN chembl.activities AS act
      ON act.assay_id = a.assay_id
    JOIN chembl.molecule_dictionary AS md
      ON md.molregno = act.molregno
    JOIN chembl.compound_structures AS cs
      ON cs.molregno = act.molregno
    WHERE d.journal IS NOT NULL
      AND TRIM(d.journal) <> ''
      AND d.doi IS NOT NULL
      AND TRIM(d.doi) <> ''
      AND md.molecule_type = 'Small molecule'
      AND cs.canonical_smiles IS NOT NULL
      AND TRIM(cs.canonical_smiles) <> ''
      AND act.standard_value IS NOT NULL;

    CREATE INDEX idx_ra_doc_assay_smiles
        ON relevant_activity(doc_id, assay_id, canonical_smiles);
    CREATE INDEX idx_ra_doi
        ON relevant_activity(doi);
    """, "Extract relevant activity rows")
else:
    print("Using cached relevant_activity table.")

summary = pd.read_sql_query("""
SELECT
    COUNT(*) AS activity_rows,
    COUNT(DISTINCT doc_id) AS depositions,
    COUNT(DISTINCT doi) AS unique_dois,
    COUNT(DISTINCT assay_id) AS assays,
    COUNT(DISTINCT canonical_smiles) AS unique_smiles_global
FROM relevant_activity
""", conn)
display(summary)


Extract relevant activity rows: 575.2 s


,activity_rows,depositions,unique_dois,assays,unique_smiles_global
0,6879158,86380,86370,1185277,1162575


## 6. De-duplicate SMILES within each deposition/assay

This is the comparison universe. If the same canonical SMILES occurs multiple times because of repeated activity rows or multiple ChEMBL molecule records, it appears only once in `assay_smiles` for that deposition/assay.


In [6]:
if not table_exists(conn, "assay_smiles"):
    timed_sql(conn, """
    CREATE TABLE assay_smiles AS
    SELECT DISTINCT
        doc_id,
        paper_chembl_id,
        doi,
        journal,
        assay_id,
        assay_chembl_id,
        canonical_smiles
    FROM relevant_activity;

    CREATE INDEX idx_as_doc_assay
        ON assay_smiles(doc_id, assay_id);
    CREATE INDEX idx_as_smiles
        ON assay_smiles(canonical_smiles);
    """, "Build unique deposition/assay/SMILES table")
else:
    print("Using cached assay_smiles table.")

print(f"Unique deposition/assay/SMILES rows: {count_rows(conn, 'assay_smiles'):,}")


Build unique deposition/assay/SMILES table: 79.7 s
Unique deposition/assay/SMILES rows: 6,650,070


## 7. Fingerprint every unique SMILES once

Fingerprinting is cached globally by the exact SMILES string for efficiency. **Collision testing is still performed only within each deposition and assay.** Invalid SMILES are retained with `parse_ok = 0` and excluded from collision testing.


In [7]:
conn.execute("""
CREATE TABLE IF NOT EXISTS smiles_fingerprints (
    canonical_smiles TEXT PRIMARY KEY,
    parse_ok INTEGER NOT NULL,
    morgan_fp BLOB,
    atompair_fp BLOB,
    torsion_fp BLOB
)
""")
conn.commit()

remaining = conn.execute("""
SELECT DISTINCT s.canonical_smiles
FROM assay_smiles AS s
LEFT JOIN smiles_fingerprints AS f
  ON f.canonical_smiles = s.canonical_smiles
WHERE f.canonical_smiles IS NULL
""").fetchall()

print(f"SMILES still to fingerprint: {len(remaining):,}")

t0 = time.time()
batch = []
for i, (smiles,) in enumerate(remaining, start=1):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        row = (smiles, 0, None, None, None)
    else:
        row = (
            smiles,
            1,
            sqlite3.Binary(fp_bytes(morgan_gen, mol)),
            sqlite3.Binary(fp_bytes(atompair_gen, mol)),
            sqlite3.Binary(fp_bytes(torsion_gen, mol)),
        )
    batch.append(row)

    if len(batch) >= FINGERPRINT_BATCH_SIZE:
        conn.executemany("""
            INSERT OR IGNORE INTO smiles_fingerprints
            (canonical_smiles, parse_ok, morgan_fp, atompair_fp, torsion_fp)
            VALUES (?, ?, ?, ?, ?)
        """, batch)
        conn.commit()
        batch.clear()
        print(f"  processed {i:,}/{len(remaining):,}")

if batch:
    conn.executemany("""
        INSERT OR IGNORE INTO smiles_fingerprints
        (canonical_smiles, parse_ok, morgan_fp, atompair_fp, torsion_fp)
        VALUES (?, ?, ?, ?, ?)
    """, batch)
    conn.commit()

if remaining:
    print(f"Fingerprinting finished in {time.time() - t0:,.1f} s")

fp_qc = pd.read_sql_query("""
SELECT parse_ok, COUNT(*) AS n_smiles
FROM smiles_fingerprints
GROUP BY parse_ok
ORDER BY parse_ok DESC
""", conn)
display(fp_qc)


SMILES still to fingerprint: 1,162,575
  processed 5,000/1,162,575
  processed 10,000/1,162,575
  processed 15,000/1,162,575
  processed 20,000/1,162,575
  processed 25,000/1,162,575
  processed 30,000/1,162,575
  processed 35,000/1,162,575
  processed 40,000/1,162,575
  processed 45,000/1,162,575
  processed 50,000/1,162,575
  processed 55,000/1,162,575
  processed 60,000/1,162,575
  processed 65,000/1,162,575
  processed 70,000/1,162,575
  processed 75,000/1,162,575
  processed 80,000/1,162,575
  processed 85,000/1,162,575
  processed 90,000/1,162,575
  processed 95,000/1,162,575
  processed 100,000/1,162,575
  processed 105,000/1,162,575
  processed 110,000/1,162,575
  processed 115,000/1,162,575
  processed 120,000/1,162,575
  processed 125,000/1,162,575
  processed 130,000/1,162,575
  processed 135,000/1,162,575
  processed 140,000/1,162,575
  processed 145,000/1,162,575
  processed 150,000/1,162,575
  processed 155,000/1,162,575
  processed 160,000/1,162,575
  processed 165,000/1

[23:03:39] Explicit valence for atom # 34 P, 7, is greater than permitted


  processed 235,000/1,162,575
  processed 240,000/1,162,575
  processed 245,000/1,162,575
  processed 250,000/1,162,575
  processed 255,000/1,162,575
  processed 260,000/1,162,575
  processed 265,000/1,162,575
  processed 270,000/1,162,575
  processed 275,000/1,162,575
  processed 280,000/1,162,575
  processed 285,000/1,162,575
  processed 290,000/1,162,575
  processed 295,000/1,162,575
  processed 300,000/1,162,575
  processed 305,000/1,162,575
  processed 310,000/1,162,575
  processed 315,000/1,162,575
  processed 320,000/1,162,575
  processed 325,000/1,162,575
  processed 330,000/1,162,575
  processed 335,000/1,162,575
  processed 340,000/1,162,575
  processed 345,000/1,162,575
  processed 350,000/1,162,575
  processed 355,000/1,162,575
  processed 360,000/1,162,575
  processed 365,000/1,162,575
  processed 370,000/1,162,575
  processed 375,000/1,162,575
  processed 380,000/1,162,575
  processed 385,000/1,162,575
  processed 390,000/1,162,575
  processed 395,000/1,162,575
  processe

[23:06:09] Can't kekulize mol.  Unkekulized atoms: 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71 72 73 74 75 76 77


  processed 505,000/1,162,575
  processed 510,000/1,162,575
  processed 515,000/1,162,575
  processed 520,000/1,162,575
  processed 525,000/1,162,575
  processed 530,000/1,162,575
  processed 535,000/1,162,575
  processed 540,000/1,162,575
  processed 545,000/1,162,575
  processed 550,000/1,162,575
  processed 555,000/1,162,575
  processed 560,000/1,162,575
  processed 565,000/1,162,575
  processed 570,000/1,162,575
  processed 575,000/1,162,575
  processed 580,000/1,162,575
  processed 585,000/1,162,575
  processed 590,000/1,162,575
  processed 595,000/1,162,575
  processed 600,000/1,162,575
  processed 605,000/1,162,575
  processed 610,000/1,162,575
  processed 615,000/1,162,575
  processed 620,000/1,162,575
  processed 625,000/1,162,575
  processed 630,000/1,162,575
  processed 635,000/1,162,575
  processed 640,000/1,162,575
  processed 645,000/1,162,575
  processed 650,000/1,162,575
  processed 655,000/1,162,575
  processed 660,000/1,162,575
  processed 665,000/1,162,575
  processe

[23:07:59] Can't kekulize mol.  Unkekulized atoms: 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62


... (8 outputs left out)

  processed 1,040,000/1,162,575
  processed 1,045,000/1,162,575
  processed 1,050,000/1,162,575
  processed 1,055,000/1,162,575
  processed 1,060,000/1,162,575
  processed 1,065,000/1,162,575
  processed 1,070,000/1,162,575
  processed 1,075,000/1,162,575
  processed 1,080,000/1,162,575
  processed 1,085,000/1,162,575
  processed 1,090,000/1,162,575
  processed 1,095,000/1,162,575
  processed 1,100,000/1,162,575
  processed 1,105,000/1,162,575
  processed 1,110,000/1,162,575
  processed 1,115,000/1,162,575
  processed 1,120,000/1,162,575
  processed 1,125,000/1,162,575
  processed 1,130,000/1,162,575
  processed 1,135,000/1,162,575
  processed 1,140,000/1,162,575
  processed 1,145,000/1,162,575


[23:11:34] WARNING: not removing hydrogen atom without neighbors
[23:11:34] WARNING: not removing hydrogen atom without neighbors


  processed 1,150,000/1,162,575
  processed 1,155,000/1,162,575
  processed 1,160,000/1,162,575
Fingerprinting finished in 638.4 s


,parse_ok,n_smiles
0,1,1162568
1,0,7


## 8. Expand to one row per assay/SMILES/fingerprint type

Storing the fingerprint type vertically makes the exact-equality collision query simple and lets SQLite use a compound index instead of comparing every possible pair in an assay.


In [8]:
if not table_exists(conn, "assay_smiles_fp"):
    timed_sql(conn, """
    CREATE TABLE assay_smiles_fp AS
    SELECT s.*, 'Morgan' AS fp_type, f.morgan_fp AS fp
    FROM assay_smiles AS s
    JOIN smiles_fingerprints AS f USING (canonical_smiles)
    WHERE f.parse_ok = 1

    UNION ALL

    SELECT s.*, 'AtomPair' AS fp_type, f.atompair_fp AS fp
    FROM assay_smiles AS s
    JOIN smiles_fingerprints AS f USING (canonical_smiles)
    WHERE f.parse_ok = 1

    UNION ALL

    SELECT s.*, 'TopologicalTorsion' AS fp_type, f.torsion_fp AS fp
    FROM assay_smiles AS s
    JOIN smiles_fingerprints AS f USING (canonical_smiles)
    WHERE f.parse_ok = 1;

    CREATE INDEX idx_asfp_collision
        ON assay_smiles_fp(fp_type, doc_id, assay_id, fp);
    """, "Build assay/SMILES/fingerprint table")
else:
    print("Using cached assay_smiles_fp table.")


Build assay/SMILES/fingerprint table: 464.3 s


## 9. Find exact fingerprint-degenerate pairs

The self-join requires the same `doc_id`, `assay_id`, fingerprint type, and exact fingerprint bytes. `a.canonical_smiles < b.canonical_smiles` ensures that the two SMILES are different and that each unordered pair appears only once.


In [9]:
if not table_exists(conn, "collision_pairs"):
    timed_sql(conn, """
    CREATE TABLE collision_pairs AS
    SELECT
        a.doc_id,
        a.paper_chembl_id,
        a.doi,
        a.journal,
        a.assay_id,
        a.assay_chembl_id,
        a.fp_type,
        a.canonical_smiles AS smiles_1,
        b.canonical_smiles AS smiles_2
    FROM assay_smiles_fp AS a
    JOIN assay_smiles_fp AS b
      ON b.fp_type = a.fp_type
     AND b.doc_id = a.doc_id
     AND b.assay_id = a.assay_id
     AND b.fp = a.fp
     AND b.canonical_smiles > a.canonical_smiles;

    CREATE INDEX idx_cp_doc_assay
        ON collision_pairs(doc_id, assay_id);
    CREATE INDEX idx_cp_fp
        ON collision_pairs(fp_type);
    """, "Find exact fingerprint-degenerate SMILES pairs")
else:
    print("Using cached collision_pairs table.")

collision_summary = pd.read_sql_query("""
SELECT
    fp_type,
    COUNT(*) AS degenerate_pairs,
    COUNT(DISTINCT doc_id) AS depositions_with_a_pair,
    COUNT(DISTINCT doi) AS papers_with_a_pair
FROM collision_pairs
GROUP BY fp_type
ORDER BY fp_type
""", conn)
display(collision_summary)


Find exact fingerprint-degenerate SMILES pairs: 92.5 s


,fp_type,degenerate_pairs,depositions_with_a_pair,papers_with_a_pair
0,AtomPair,97668,9014,9013
1,Morgan,158234,6703,6703
2,TopologicalTorsion,161519,10229,10229


## 10. Pull only activity rows belonging to collision SMILES

For activity comparisons, rows are restricted to exact measurements (`standard_relation = '='`). We also require both `standard_type` and `standard_units`, because values with different endpoints or units should not be directly subtracted.


In [10]:
if not table_exists(conn, "collision_activity"):
    timed_sql(conn, """
    CREATE TABLE collision_activity AS
    WITH collision_smiles AS (
        SELECT DISTINCT doc_id, assay_id, smiles_1 AS canonical_smiles
        FROM collision_pairs
        UNION
        SELECT DISTINCT doc_id, assay_id, smiles_2 AS canonical_smiles
        FROM collision_pairs
    )
    SELECT r.*
    FROM relevant_activity AS r
    JOIN collision_smiles AS c
      ON c.doc_id = r.doc_id
     AND c.assay_id = r.assay_id
     AND c.canonical_smiles = r.canonical_smiles
    WHERE TRIM(COALESCE(r.standard_relation, '')) = '='
      AND r.standard_type IS NOT NULL
      AND TRIM(r.standard_type) <> ''
      AND r.standard_units IS NOT NULL
      AND TRIM(r.standard_units) <> ''
      AND r.standard_value IS NOT NULL;

    CREATE INDEX idx_ca_endpoint
        ON collision_activity(doc_id, assay_id, canonical_smiles, standard_type, standard_units);
    """, "Extract comparable activities for collision SMILES")
else:
    print("Using cached collision_activity table.")

print(f"Comparable activity rows for collision SMILES: {count_rows(conn, 'collision_activity'):,}")


Extract comparable activities for collision SMILES: 6.5 s
Comparable activity rows for collision SMILES: 270,829


## 11. Collapse activity replicates and attach them to the degenerate pairs

For each deposition/assay/SMILES/endpoint, the representative activity is the **median** standardized value. A pair is comparable only if both SMILES have a representative value for exactly the same `standard_type` and `standard_units`.


In [11]:
activity = pd.read_sql_query("""
SELECT
    doc_id, assay_id, canonical_smiles,
    standard_type, standard_units,
    activity_id, standard_value
FROM collision_activity
""", conn)

activity["standard_value"] = pd.to_numeric(activity["standard_value"], errors="coerce")
activity = activity.dropna(subset=["standard_value"])

rep = (
    activity
    .groupby(
        ["doc_id", "assay_id", "canonical_smiles", "standard_type", "standard_units"],
        as_index=False,
        dropna=False,
    )
    .agg(
        activity_value=("standard_value", "median"),
        n_activity_rows=("activity_id", "nunique"),
        min_activity=("standard_value", "min"),
        max_activity=("standard_value", "max"),
    )
)

pairs = pd.read_sql_query("SELECT * FROM collision_pairs", conn)

left = rep.rename(columns={
    "canonical_smiles": "smiles_1",
    "activity_value": "activity_1",
    "n_activity_rows": "n_activity_rows_1",
    "min_activity": "min_activity_1",
    "max_activity": "max_activity_1",
})
right = rep.rename(columns={
    "canonical_smiles": "smiles_2",
    "activity_value": "activity_2",
    "n_activity_rows": "n_activity_rows_2",
    "min_activity": "min_activity_2",
    "max_activity": "max_activity_2",
})

pair_activity = pairs.merge(
    left,
    on=["doc_id", "assay_id", "smiles_1"],
    how="inner",
)
pair_activity = pair_activity.merge(
    right,
    on=["doc_id", "assay_id", "smiles_2", "standard_type", "standard_units"],
    how="inner",
)

pair_activity["absolute_activity_difference"] = (
    pair_activity["activity_1"] - pair_activity["activity_2"]
).abs()

positive = (pair_activity["activity_1"] > 0) & (pair_activity["activity_2"] > 0)
pair_activity["fold_difference"] = np.nan
pair_activity.loc[positive, "fold_difference"] = (
    pair_activity.loc[positive, ["activity_1", "activity_2"]].max(axis=1)
    / pair_activity.loc[positive, ["activity_1", "activity_2"]].min(axis=1)
)

pair_activity["has_activity_difference"] = ~np.isclose(
    pair_activity["activity_1"].to_numpy(dtype=float),
    pair_activity["activity_2"].to_numpy(dtype=float),
    rtol=1e-12,
    atol=0.0,
    equal_nan=False,
)

degenerate_with_difference = pair_activity.loc[
    pair_activity["has_activity_difference"]
].copy()

print(f"Degenerate pair/endpoint comparisons: {len(pair_activity):,}")
print(f"...with a non-zero activity difference: {len(degenerate_with_difference):,}")
display(degenerate_with_difference.head(20))


Degenerate pair/endpoint comparisons: 300,598
...with a non-zero activity difference: 273,721


,doc_id,paper_chembl_id,doi,journal,assay_id,assay_chembl_id,fp_type,smiles_1,smiles_2,standard_type,standard_units,activity_1,n_activity_rows_1,min_activity_1,max_activity_1,activity_2,n_activity_rows_2,min_activity_2,max_activity_2,absolute_activity_difference,fold_difference,has_activity_difference
0,10,CHEMBL1148467,10.1021/jm0301888,J Med Chem,5296,CHEMBL618377,Morgan,O=C(O)Cc1ccccc1OCCC1Oc2ccccc2N(CCCCCCCO)C1=O,O=C(O)Cc1ccccc1OCCC1Oc2ccccc2N(CCCCCCO)C1=O,T1/2,hr,1.2670,1,1.267,1.267,0.950,1,0.950,0.950,0.3170,1.333684,True
1,10,CHEMBL1148467,10.1021/jm0301888,J Med Chem,9293,CHEMBL631322,Morgan,O=C(O)Cc1ccccc1OCCC1Oc2ccccc2N(CCCCCCCO)C1=O,O=C(O)Cc1ccccc1OCCC1Oc2ccccc2N(CCCCCCO)C1=O,AUC,ng.hr.mL-1,1015.5000,1,1015.500,1015.500,1068.750,1,1068.750,1068.750,53.2500,1.052437,True
2,10,CHEMBL1148467,10.1021/jm0301888,J Med Chem,9825,CHEMBL622838,Morgan,O=C(O)Cc1ccccc1OCCC1Oc2ccccc2N(CCCCCCCO)C1=O,O=C(O)Cc1ccccc1OCCC1Oc2ccccc2N(CCCCCCO)C1=O,T1/2,hr,7.0000,1,7.000,7.000,0.600,1,0.600,0.600,6.4000,11.666667,True
3,10,CHEMBL1148467,10.1021/jm0301888,J Med Chem,10462,CHEMBL621847,Morgan,O=C(O)Cc1ccccc1OCCC1Oc2ccccc2N(CCCCCCCO)C1=O,O=C(O)Cc1ccccc1OCCC1Oc2ccccc2N(CCCCCCO)C1=O,F,%,13.0000,1,13.000,13.000,4.000,1,4.000,4.000,9.0000,3.250000,True
4,10,CHEMBL1148467,10.1021/jm0301888,J Med Chem,154042,CHEMBL757757,Morgan,CCCCCCCCCCN1C(=O)C(CCOc2ccccc2CC(=O)O)Oc2ccccc21,CCCCCCCCN1C(=O)C(CCOc2ccccc2CC(=O)O)Oc2ccccc21,EC50,nM,1000.0000,1,1000.000,1000.000,300.000,1,300.000,300.000,700.0000,3.333333,True
5,10,CHEMBL1148467,10.1021/jm0301888,J Med Chem,154042,CHEMBL757757,Morgan,CCCCCCCCCCN1C(=O)C(CCOc2ccccc2CC(=O)O)Oc2ccccc21,CCCCCCCN1C(=O)C(CCOc2ccccc2CC(=O)O)Oc2ccccc21,EC50,nM,1000.0000,1,1000.000,1000.000,234.000,1,234.000,234.000,766.0000,4.273504,True
6,10,CHEMBL1148467,10.1021/jm0301888,J Med Chem,154042,CHEMBL757757,Morgan,CCCCCCCCCCN1C(=O)C(CCOc2ccccc2CC(=O)O)Oc2ccccc21,CCCCCCN1C(=O)C(CCOc2ccccc2CC(=O)O)Oc2ccccc21,EC50,nM,1000.0000,1,1000.000,1000.000,100.000,1,100.000,100.000,900.0000,10.000000,True
7,10,CHEMBL1148467,10.1021/jm0301888,J Med Chem,154042,CHEMBL757757,Morgan,CCCCCCCCN1C(=O)C(CCOc2ccccc2CC(=O)O)Oc2ccccc21,CCCCCCCN1C(=O)C(CCOc2ccccc2CC(=O)O)Oc2ccccc21,EC50,nM,300.0000,1,300.000,300.000,234.000,1,234.000,234.000,66.0000,1.282051,True
8,10,CHEMBL1148467,10.1021/jm0301888,J Med Chem,154042,CHEMBL757757,Morgan,CCCCCCCCN1C(=O)C(CCOc2ccccc2CC(=O)O)Oc2ccccc21,CCCCCCN1C(=O)C(CCOc2ccccc2CC(=O)O)Oc2ccccc21,EC50,nM,300.0000,1,300.000,300.000,100.000,1,100.000,100.000,200.0000,3.000000,True
9,10,CHEMBL1148467,10.1021/jm0301888,J Med Chem,154042,CHEMBL757757,Morgan,CCCCCCCN1C(=O)C(CCOc2ccccc2CC(=O)O)Oc2ccccc21,CCCCCCN1C(=O)C(CCOc2ccccc2CC(=O)O)Oc2ccccc21,EC50,nM,234.0000,1,234.000,234.000,100.000,1,100.000,100.000,134.0000,2.340000,True



## 12. Collapse pairwise comparisons into exact fingerprint groups

The pairwise table is retained internally because it already enforces the activity-comparison rules above. For the CSV output, however, molecules are grouped by the **exact fingerprint bytes within the same deposition, assay, and fingerprint type**. Each degenerate fingerprint group therefore appears once.

`group_size` is the number of distinct canonical SMILES in the group. `max_fold_difference` is the largest valid pairwise fold difference anywhere in that group, considering only comparisons with the same `standard_type` and `standard_units` and positive representative activity values, exactly as above. The SMILES, endpoint, units, and activity values responsible for that maximum are retained for auditability.


In [12]:

import json

# Recover the actual fingerprint groups directly from assay_smiles_fp.
# This is preferable to reconstructing groups from overlapping collision pairs.
group_members = pd.read_sql_query("""
WITH collision_groups AS (
    SELECT
        doc_id,
        assay_id,
        fp_type,
        fp,
        COUNT(DISTINCT canonical_smiles) AS group_size
    FROM assay_smiles_fp
    GROUP BY doc_id, assay_id, fp_type, fp
    HAVING COUNT(DISTINCT canonical_smiles) > 1
)
SELECT
    a.doc_id,
    a.paper_chembl_id,
    a.doi,
    a.journal,
    a.assay_id,
    a.assay_chembl_id,
    a.fp_type,
    HEX(a.fp) AS fingerprint_hex,
    a.canonical_smiles
FROM assay_smiles_fp AS a
JOIN collision_groups AS g
  ON g.doc_id = a.doc_id
 AND g.assay_id = a.assay_id
 AND g.fp_type = a.fp_type
 AND g.fp = a.fp
ORDER BY a.doc_id, a.assay_id, a.fp_type, fingerprint_hex, a.canonical_smiles
""", conn)

# Give each exact fingerprint group a compact deterministic integer ID.
group_keys = (
    group_members[["doc_id", "assay_id", "fp_type", "fingerprint_hex"]]
    .drop_duplicates()
    .sort_values(["doc_id", "assay_id", "fp_type", "fingerprint_hex"])
    .reset_index(drop=True)
)
group_keys["group_id"] = np.arange(1, len(group_keys) + 1)

group_members = group_members.merge(
    group_keys,
    on=["doc_id", "assay_id", "fp_type", "fingerprint_hex"],
    how="left",
    validate="many_to_one",
)

# One base row per fingerprint group. Store all member SMILES as a JSON array so
# the field remains unambiguous when written inside a CSV cell.
group_summary = (
    group_members
    .groupby(
        [
            "group_id", "doc_id", "paper_chembl_id", "doi", "journal",
            "assay_id", "assay_chembl_id", "fp_type", "fingerprint_hex",
        ],
        as_index=False,
        dropna=False,
    )
    .agg(
        group_size=("canonical_smiles", "nunique"),
        member_smiles=(
            "canonical_smiles",
            lambda x: json.dumps(sorted(set(x)), ensure_ascii=False),
        ),
    )
)

# Attach every comparable pair/endpoint row to its exact fingerprint group.
smiles_to_group = (
    group_members[["doc_id", "assay_id", "fp_type", "canonical_smiles", "group_id"]]
    .drop_duplicates()
    .rename(columns={"canonical_smiles": "smiles_1"})
)

pair_activity_grouped = pair_activity.merge(
    smiles_to_group,
    on=["doc_id", "assay_id", "fp_type", "smiles_1"],
    how="inner",
    validate="many_to_one",
)

# Group-level comparison counts and whether any comparable pair differs.
comparison_stats = (
    pair_activity_grouped
    .groupby("group_id", as_index=False)
    .agg(
        n_comparable_pair_endpoint_comparisons=("has_activity_difference", "size"),
        has_any_activity_difference=("has_activity_difference", "any"),
        max_absolute_activity_difference=("absolute_activity_difference", "max"),
    )
)

distinct_pair_counts = (
    pair_activity_grouped[["group_id", "smiles_1", "smiles_2"]]
    .drop_duplicates()
    .groupby("group_id")
    .size()
    .rename("n_distinct_comparable_pairs")
    .reset_index()
)

comparison_stats = comparison_stats.merge(
    distinct_pair_counts,
    on="group_id",
    how="left",
    validate="one_to_one",
)

group_summary = group_summary.merge(
    comparison_stats,
    on="group_id",
    how="left",
    validate="one_to_one",
)

group_summary["n_comparable_pair_endpoint_comparisons"] = (
    group_summary["n_comparable_pair_endpoint_comparisons"].fillna(0).astype(int)
)
group_summary["n_distinct_comparable_pairs"] = (
    group_summary["n_distinct_comparable_pairs"].fillna(0).astype(int)
)
group_summary["has_any_activity_difference"] = (
    group_summary["has_any_activity_difference"].fillna(False).astype(bool)
)

# Find the single pair/endpoint row giving the largest valid fold difference
# within each fingerprint group. idxmax chooses one row if there is an exact tie.
valid_fold = pair_activity_grouped.loc[
    pair_activity_grouped["fold_difference"].notna()
].copy()

max_fold_cols = [
    "max_fold_difference",
    "max_fold_smiles_1",
    "max_fold_smiles_2",
    "max_fold_standard_type",
    "max_fold_standard_units",
    "max_fold_activity_1",
    "max_fold_activity_2",
]

if len(valid_fold):
    max_idx = valid_fold.groupby("group_id")["fold_difference"].idxmax()
    max_fold_details = valid_fold.loc[
        max_idx,
        [
            "group_id", "fold_difference", "smiles_1", "smiles_2",
            "standard_type", "standard_units", "activity_1", "activity_2",
        ],
    ].rename(columns={
        "fold_difference": "max_fold_difference",
        "smiles_1": "max_fold_smiles_1",
        "smiles_2": "max_fold_smiles_2",
        "standard_type": "max_fold_standard_type",
        "standard_units": "max_fold_standard_units",
        "activity_1": "max_fold_activity_1",
        "activity_2": "max_fold_activity_2",
    })

    group_summary = group_summary.merge(
        max_fold_details,
        on="group_id",
        how="left",
        validate="one_to_one",
    )
else:
    for col in max_fold_cols:
        group_summary[col] = np.nan

# Convenient subset corresponding to groups with at least one non-zero
# comparable activity difference.
degenerate_groups_with_difference = group_summary.loc[
    group_summary["has_any_activity_difference"]
].copy()

print(f"Exact degenerate fingerprint groups: {len(group_summary):,}")
print(
    "...with at least one comparable non-zero activity difference: "
    f"{len(degenerate_groups_with_difference):,}"
)
display(group_summary.head(20))


C:\Users\acebl\AppData\Local\Temp\ipykernel_27688\2127211744.py:128: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  group_summary["has_any_activity_difference"].fillna(False).astype(bool)


Exact degenerate fingerprint groups: 218,320
...with at least one comparable non-zero activity difference: 146,699


,group_id,doc_id,paper_chembl_id,doi,journal,assay_id,assay_chembl_id,fp_type,fingerprint_hex,group_size,member_smiles,n_comparable_pair_endpoint_comparisons,has_any_activity_difference,max_absolute_activity_difference,n_distinct_comparable_pairs,max_fold_difference,max_fold_smiles_1,max_fold_smiles_2,max_fold_standard_type,max_fold_standard_units,max_fold_activity_1,max_fold_activity_2
0,1,8,CHEMBL1139457,10.1021/jm0309811,J Med Chem,2670,CHEMBL617918,AtomPair,7F00031131300333010000013100003300100000003030...,2,"[""CN1CCN([C@@H]2Cc3ccccc3Sc3ccc(Cl)cc32)CC1"", ...",1,True,0.1900,1,2.357143,CN1CCN([C@@H]2Cc3ccccc3Sc3ccc(Cl)cc32)CC1,CN1CCN([C@H]2Cc3ccccc3Sc3ccc(Cl)cc32)CC1,Ki,nM,0.3300,0.14
1,2,8,CHEMBL1139457,10.1021/jm0309811,J Med Chem,21573,CHEMBL857070,AtomPair,7F00031131300333010000013100003300100000003030...,2,"[""CN1CCN([C@@H]2Cc3ccccc3Sc3ccc(Cl)cc32)CC1"", ...",0,False,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,8,CHEMBL1139457,10.1021/jm0309811,J Med Chem,58674,CHEMBL670448,AtomPair,7F00031131300333010000013100003300100000003030...,2,"[""CN1CCN([C@@H]2Cc3ccccc3Sc3ccc(Cl)cc32)CC1"", ...",1,True,0.1000,1,1.052632,CN1CCN([C@@H]2Cc3ccccc3Sc3ccc(Cl)cc32)CC1,CN1CCN([C@H]2Cc3ccccc3Sc3ccc(Cl)cc32)CC1,Ki,nM,2.0000,1.90
3,4,8,CHEMBL1139457,10.1021/jm0309811,J Med Chem,62769,CHEMBL676296,AtomPair,7F00031131300333010000013100003300100000003030...,2,"[""CN1CCN([C@@H]2Cc3ccccc3Sc3ccc(Cl)cc32)CC1"", ...",1,True,20.6000,1,52.500000,CN1CCN([C@@H]2Cc3ccccc3Sc3ccc(Cl)cc32)CC1,CN1CCN([C@H]2Cc3ccccc3Sc3ccc(Cl)cc32)CC1,Ki,nM,21.0000,0.40
4,5,8,CHEMBL1139457,10.1021/jm0309811,J Med Chem,62880,CHEMBL673590,AtomPair,7F00031131300333010000013100003300100000003030...,2,"[""CN1CCN([C@@H]2Cc3ccccc3Sc3ccc(Cl)cc32)CC1"", ...",1,True,3.2000,1,9.000000,CN1CCN([C@@H]2Cc3ccccc3Sc3ccc(Cl)cc32)CC1,CN1CCN([C@H]2Cc3ccccc3Sc3ccc(Cl)cc32)CC1,Ki,nM,3.6000,0.40
5,6,8,CHEMBL1139457,10.1021/jm0309811,J Med Chem,229510,CHEMBL852023,AtomPair,7F00031131300333010000013100003300100000003030...,2,"[""CN1CCN([C@@H]2Cc3ccccc3Sc3ccc(Cl)cc32)CC1"", ...",0,False,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,7,9,CHEMBL1139458,10.1021/jm034206x,J Med Chem,84985,CHEMBL695719,AtomPair,3013077100130313110030700103011001000010000000...,2,"[""C[C@@H](NC(=O)c1cccc(-c2ccc(S(=O)(=O)CC(=O)O...",1,True,1657.0000,1,39.534884,C[C@@H](NC(=O)c1cccc(-c2ccc(S(=O)(=O)CC(=O)O)c...,C[C@H](NC(=O)c1cccc(-c2ccc(S(=O)(=O)CC(=O)O)cc...,IC50,nM,43.0000,1700.00
7,8,9,CHEMBL1139458,10.1021/jm034206x,J Med Chem,84985,CHEMBL695719,AtomPair,3713077310730113300010300303310010010011000010...,2,"[""C[C@@H](NC(=O)Cc1cccc(-c2ccc(S(=O)(=O)CC(=O)...",1,True,233.0000,1,3.678161,C[C@@H](NC(=O)Cc1cccc(-c2ccc(S(=O)(=O)CC(=O)O)...,C[C@H](NC(=O)Cc1cccc(-c2ccc(S(=O)(=O)CC(=O)O)c...,IC50,nM,320.0000,87.00
8,9,10,CHEMBL1148467,10.1021/jm0301888,J Med Chem,5296,CHEMBL618377,Morgan,2020000000001000010001000044D00200000000008080...,2,"[""O=C(O)Cc1ccccc1OCCC1Oc2ccccc2N(CCCCCCCO)C1=O...",1,True,0.3170,1,1.333684,O=C(O)Cc1ccccc1OCCC1Oc2ccccc2N(CCCCCCCO)C1=O,O=C(O)Cc1ccccc1OCCC1Oc2ccccc2N(CCCCCCO)C1=O,T1/2,hr,1.2670,0.95
9,10,10,CHEMBL1148467,10.1021/jm0301888,J Med Chem,5296,CHEMBL618377,TopologicalTorsion,0130003000000700000000000000000000000001000000...,2,"[""CCCCCCN1C(=O)C(CCOc2ccccc2CC(=O)O)Oc2ccccc21...",1,True,0.0667,1,1.075512,CCCCCCN1C(=O)C(CCOc2ccccc2CC(=O)O)Oc2ccccc21,O=C(O)Cc1ccccc1OCCC1Oc2ccccc2N(CCCCCCO)C1=O,T1/2,hr,0.8833,0.95


## 13. Count papers for the final chart

A paper is identified by its DOI string for this final count. The same DOI is counted once per fingerprint type even if it has many assays, collision pairs, or activity endpoints. `Total / any FP` is the number of unique DOIs occurring in the union of all three fingerprint-specific result sets.


In [13]:
counts = (
    degenerate_groups_with_difference
    .groupby("fp_type")["doi"]
    .nunique()
    .reindex(FP_NAMES, fill_value=0)
)

total_any_fp = degenerate_groups_with_difference["doi"].nunique()

plot_data = pd.DataFrame({
    "Fingerprint": FP_NAMES + ["Total / any FP"],
    "Papers": [int(counts[x]) for x in FP_NAMES] + [int(total_any_fp)],
})

display(plot_data)


,Fingerprint,Papers
0,Morgan,6055
1,AtomPair,7988
2,TopologicalTorsion,9068
3,Total / any FP,15329


## 14. Plotly bar chart


In [17]:
import plotly.express as px

# Rename fingerprint labels for presentation
plot_data = plot_data.copy()

plot_data["Fingerprint"] = plot_data["Fingerprint"].replace({
    "Morgan": "ECFP4",
    "AtomPair": "Atom Pair",
    "TopologicalTorsion": "Torsion",
    "Total / any FP": "Total",
    "Total/any FP": "Total",
    "Any fingerprint": "Total",
})

# Bar fill colors with alpha = 0.18
bar_colors = {
    "ECFP4": "rgba(59, 110, 168, 0.18)",     # #3B6EA8
    "Atom Pair": "rgba(111, 168, 182, 0.18)", # #6FA8B6
    "Torsion": "rgba(167, 173, 178, 0.18)",   # #A7ADB2
    "Total": "rgba(31, 58, 95, 0.18)",        # #1F3A5F
}

fig = px.bar(
    plot_data,
    x="Fingerprint",
    y="Papers",
    text="Papers",
)

fig.update_traces(
    texttemplate="%{y:,}",
    textposition="outside",
    cliponaxis=False,
    marker=dict(
        color=[bar_colors[x] for x in plot_data["Fingerprint"]],
        line=dict(
            color="black",
            width=1,
        ),
    ),
)

fig.update_layout(
    width=500,
    height=450,

    # Transparent background
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",

    # Axes
    xaxis_title=None,
    yaxis_title="Number of papers",

    # Bar spacing
    bargap=0.35,

    # Clean appearance
    showlegend=False,
    margin=dict(l=70, r=20, t=30, b=60),

    font=dict(
        size=14,
        family="Arial",
    ),
)

fig.update_xaxes(
    showgrid=False,
    showline=True,
    linewidth=1,
    linecolor="black",
    ticks="outside",
)

fig.update_yaxes(
    tickformat=",",
    showgrid=False,
    showline=True,
    linewidth=1,
    linecolor="black",
    ticks="outside",
    rangemode="tozero",
)

fig.show()

# Export as vector PDF
fig.write_image(
    "fingerprint_degeneracy_papers.pdf",
    format="pdf",
    width=500,
    height=450,
)

## 15. Export results


In [15]:

# Main outputs: one row per exact fingerprint-degenerate group.
group_summary.to_csv(
    OUTPUT_DIR / "degenerate_fingerprint_groups.csv",
    index=False,
)
degenerate_groups_with_difference.to_csv(
    OUTPUT_DIR / "degenerate_fingerprint_groups_with_activity_difference.csv",
    index=False,
)

plot_data.to_csv(OUTPUT_DIR / "paper_counts_by_fingerprint.csv", index=False)
fig.write_html(OUTPUT_DIR / "paper_counts_by_fingerprint.html", include_plotlyjs="cdn")

# One row per DOI/fingerprint type that qualifies for the chart.
paper_fp_flags = (
    degenerate_groups_with_difference[["doi", "fp_type"]]
    .drop_duplicates()
    .sort_values(["doi", "fp_type"])
)
paper_fp_flags.to_csv(OUTPUT_DIR / "qualifying_papers_by_fingerprint.csv", index=False)

print("Wrote results to:", OUTPUT_DIR)


Wrote results to: C:\Users\acebl\Downloads\chembl_degeneracy_results_grouped


## 16. Optional sanity checks

These are useful for auditing the final counts. In particular, the first table shows the number of qualifying papers, assays, and degenerate pairs per fingerprint type; the second shows how many qualifying fingerprint types each paper appears in.


In [18]:
if len(degenerate_groups_with_difference):
    audit = (
        degenerate_groups_with_difference
        .groupby("fp_type")
        .agg(
            papers=("doi", "nunique"),
            depositions=("doc_id", "nunique"),
            assays=("assay_id", "nunique"),
            degenerate_groups=("group_id", "nunique"),
        )
        .reindex(FP_NAMES)
    )
    display(audit)

    overlap = (
        paper_fp_flags.groupby("doi")["fp_type"]
        .nunique()
        .value_counts()
        .sort_index()
        .rename_axis("number_of_FP_types")
        .to_frame("papers")
    )
    display(overlap)
else:
    print("No degenerate fingerprint groups with comparable non-zero activity differences were found.")


,papers,depositions,assays,degenerate_groups
fp_type,,,,
Morgan,6055,6055,21329,38041
AtomPair,7988,7989,25679,51112
TopologicalTorsion,9068,9068,29770,57546


,papers
number_of_FP_types,
1,8543
2,5790
3,996


In [19]:
# Morgan-only distribution of group-level maximum fold changes
morgan_groups = group_summary.loc[
    (group_summary["fp_type"] == "Morgan")
    & group_summary["max_fold_difference"].notna()
].copy()

fold_labels = ["<2×", "2–<10×", "10–<100×", "≥100×"]

morgan_groups["fold_bin"] = pd.cut(
    morgan_groups["max_fold_difference"],
    bins=[0, 2, 10, 100, np.inf],
    labels=fold_labels,
    right=False,
    include_lowest=True,
)

morgan_fold_counts = (
    morgan_groups["fold_bin"]
    .value_counts(sort=False)
    .reindex(fold_labels, fill_value=0)
    .rename_axis("Maximum fold difference")
    .reset_index(name="Number of groups")
)

display(morgan_fold_counts)

fig_morgan_fold = px.bar(
    morgan_fold_counts,
    x="Maximum fold difference",
    y="Number of groups",
    text="Number of groups",
)

fig_morgan_fold.update_traces(
    texttemplate="%{y:,}",
    textposition="outside",
    cliponaxis=False,
    marker=dict(
        color="rgba(59, 110, 168, 0.18)",
        line=dict(color="black", width=1),
    ),
)

fig_morgan_fold.update_layout(
    width=550,
    height=450,
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    xaxis_title="Maximum fold difference within Morgan fingerprint group",
    yaxis_title="Number of groups",
    bargap=0.35,
    showlegend=False,
    margin=dict(l=75, r=20, t=30, b=80),
    font=dict(size=14, family="Arial"),
)

fig_morgan_fold.update_xaxes(
    showgrid=False,
    showline=True,
    linewidth=1,
    linecolor="black",
    ticks="outside",
)

fig_morgan_fold.update_yaxes(
    tickformat=",",
    showgrid=False,
    showline=True,
    linewidth=1,
    linecolor="black",
    ticks="outside",
    rangemode="tozero",
)

fig_morgan_fold.show()

# Save the binned counts and plot alongside the other notebook outputs.
morgan_fold_counts.to_csv(
    OUTPUT_DIR / "morgan_group_counts_by_max_fold.csv",
    index=False,
)
fig_morgan_fold.write_html(
    OUTPUT_DIR / "morgan_group_counts_by_max_fold.html",
    include_plotlyjs="cdn",
)
fig_morgan_fold.write_image(
    OUTPUT_DIR / "morgan_group_counts_by_max_fold.pdf",
    format="pdf",
    width=550,
    height=450,
)


,Maximum fold difference,Number of groups
0,<2×,19174
1,2–<10×,14981
2,10–<100×,4519
3,≥100×,909
